# 01 — Bronze Layer: Ingestão das Fontes Brutas

Ingere todas as fontes preservando os dados como recebidos. Cada fonte é salva como Delta Table em `sandbox.bronze.*`.

| Arquivo | Formato | Tabela |
|---|---|---|
| erp_pedidos_cabecalho_2025.csv | CSV `;` | bronze.erp_pedidos_cabecalho |
| erp_pedidos_itens_2025.csv | CSV `,` | bronze.erp_pedidos_itens |
| crm_clientes_export.xlsx | XLSX | bronze.crm_clientes |
| comercial_canais.xlsx | XLSX | bronze.comercial_canais |
| cadastro_produtos_api_dump.json | JSON aninhado | bronze.cadastro_produtos |
| logistica_entregas.json | JSON aninhado | bronze.logistica_entregas |
| atendimento_ocorrencias.ndjson | NDJSON | bronze.atendimento_ocorrencias |
| vendedores.csv | CSV `;` | bronze.vendedores |
| legado_regioes_pipe.txt | Pipe | bronze.legado_regioes |

## Setup

In [0]:
from pyspark.sql.functions import current_timestamp, lit
import pandas as pd

# Unity Catalog: usar catálogo sandbox
spark.sql("USE CATALOG sandbox")

# Caminho do volume com os arquivos fonte
BASE_PATH = "/Volumes/sandbox/case_de/sources"

In [0]:
pip install openpyxl

## 0. Criação do schema bronze

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze COMMENT 'Camada Bronze — dados brutos ingeridos sem transformação'")
print("schema sandbox.bronze pronto")

## 1. erp_pedidos_cabecalho_2025.csv

In [0]:
df_ped_cab = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("multiLine", "true")
    .option("escape", '"')
    .csv(f"{BASE_PATH}/erp_pedidos_cabecalho_2025.csv")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit("erp_pedidos_cabecalho_2025.csv"))
)

df_ped_cab.write.format("delta").mode("overwrite").saveAsTable("bronze.erp_pedidos_cabecalho")
print(f"bronze.erp_pedidos_cabecalho: {df_ped_cab.count()} registros")
df_ped_cab.printSchema()

## 2. erp_pedidos_itens_2025.csv

In [0]:
df_ped_itens = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .csv(f"{BASE_PATH}/erp_pedidos_itens_2025.csv")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit("erp_pedidos_itens_2025.csv"))
)

df_ped_itens.write.format("delta").mode("overwrite").saveAsTable("bronze.erp_pedidos_itens")
print(f"bronze.erp_pedidos_itens: {df_ped_itens.count()} registros")
df_ped_itens.printSchema()

## 3. crm_clientes_export.xlsx

XLSX não tem suporte nativo no Spark — leitura via pandas e conversão para DataFrame Spark. No Unity Catalog, o caminho do volume é acessível diretamente (sem `/dbfs/`).

In [0]:
pdf_clientes = pd.read_excel(f"{BASE_PATH}/crm_clientes_export.xlsx", dtype=str)
pdf_clientes["_ingested_at"] = pd.Timestamp.now()
pdf_clientes["_source_file"] = "crm_clientes_export.xlsx"

df_clientes = spark.createDataFrame(pdf_clientes)
df_clientes.write.format("delta").mode("overwrite").saveAsTable("bronze.crm_clientes")
print(f"bronze.crm_clientes: {df_clientes.count()} registros")
df_clientes.printSchema()

## 4. comercial_canais.xlsx

In [0]:
pdf_canais = pd.read_excel(f"{BASE_PATH}/comercial_canais.xlsx", dtype=str)
pdf_canais["_ingested_at"] = pd.Timestamp.now()
pdf_canais["_source_file"] = "comercial_canais.xlsx"

df_canais = spark.createDataFrame(pdf_canais)
df_canais.write.format("delta").mode("overwrite").saveAsTable("bronze.comercial_canais")
print(f"bronze.comercial_canais: {df_canais.count()} registros")
df_canais.show(truncate=False)

## 5. cadastro_produtos_api_dump.json

Array JSON com subobjetos aninhados (`product`, `pricing`, `attributes`) — requer `multiLine`.

In [0]:
df_produtos_raw = (
    spark.read
    .option("multiLine", "true")
    .json(f"{BASE_PATH}/cadastro_produtos_api_dump.json")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit("cadastro_produtos_api_dump.json"))
)

df_produtos_raw.write.format("delta").mode("overwrite").saveAsTable("bronze.cadastro_produtos")
print(f"bronze.cadastro_produtos: {df_produtos_raw.count()} registros")
df_produtos_raw.printSchema()

## 6. logistica_entregas.json

In [0]:
df_entregas_raw = (
    spark.read
    .option("multiLine", "true")
    .json(f"{BASE_PATH}/logistica_entregas.json")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit("logistica_entregas.json"))
)

df_entregas_raw.write.format("delta").mode("overwrite").saveAsTable("bronze.logistica_entregas")
print(f"bronze.logistica_entregas: {df_entregas_raw.count()} registros")
df_entregas_raw.printSchema()

## 7. atendimento_ocorrencias.ndjson

NDJSON = um objeto JSON por linha. `spark.read.json` lê nativamente sem `multiLine`.

In [0]:
df_ocorrencias = (
    spark.read
    .json(f"{BASE_PATH}/atendimento_ocorrencias.ndjson")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit("atendimento_ocorrencias.ndjson"))
)

df_ocorrencias.write.format("delta").mode("overwrite").saveAsTable("bronze.atendimento_ocorrencias")
print(f"bronze.atendimento_ocorrencias: {df_ocorrencias.count()} registros")
df_ocorrencias.printSchema()

## 8. vendedores.csv

In [0]:
df_vendedores = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .csv(f"{BASE_PATH}/vendedores.csv")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit("vendedores.csv"))
)

df_vendedores.write.format("delta").mode("overwrite").saveAsTable("bronze.vendedores")
print(f"bronze.vendedores: {df_vendedores.count()} registros")
df_vendedores.show(truncate=False)

## 9. legado_regioes_pipe.txt

In [0]:
df_regioes = (
    spark.read
    .option("header", "true")
    .option("sep", "|")
    .csv(f"{BASE_PATH}/legado_regioes_pipe.txt")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit("legado_regioes_pipe.txt"))
)

df_regioes.write.format("delta").mode("overwrite").saveAsTable("bronze.legado_regioes")
print(f"bronze.legado_regioes: {df_regioes.count()} registros")
df_regioes.show(truncate=False)

## Validação Final — Contagem por Tabela

In [0]:
tables = [
    "bronze.erp_pedidos_cabecalho",
    "bronze.erp_pedidos_itens",
    "bronze.crm_clientes",
    "bronze.comercial_canais",
    "bronze.cadastro_produtos",
    "bronze.logistica_entregas",
    "bronze.atendimento_ocorrencias",
    "bronze.vendedores",
    "bronze.legado_regioes",
]

print("=" * 55)
print("RESUMO — BRONZE LAYER (sandbox.bronze.*)")
print("=" * 55)
for t in tables:
    count = spark.table(t).count()
    print(f"  {t:<45} {count:>6} registros")